# Trinity Fused MoE TKG with Expert Bias — End-to-End Validation

## Background

Trinity models use a **Mixture of Experts (MoE)** architecture with 128 experts and top-8 routing. The routing uses a post-sigmoid additive bias called `expert_bias` that influences **which** experts are selected, but does NOT affect the routing weight values:

```python
logits = router_linear(hidden_states)          # [batch, num_experts]
scores = sigmoid(logits)                       # sigmoid activation
selection_scores = scores + expert_bias        # bias for SELECTION only
top_k_indices = topk(selection_scores, k=8)    # select experts using biased scores
routing_weights = scores[top_k_indices]        # gather UNBIASED scores for weights
```

## The Problem

The NxDI **fused MoE TKG kernel** (`moe_token_gen_selective_load_kernel`) runs the router, expert MLPs, shared experts, and RMSNorm in a single mega-kernel — eliminating ~162 HBM round-trips per forward pass (3 per MoE layer × 54 layers). This gives a **27.7% throughput improvement** on Trinity-Mini.

However, the fused kernel's internal `router_topk` NKI function does NOT support `expert_bias`. Without it:
- Expert selection is incorrect (different experts chosen)
- Output is garbled ("the with with with withings..." instead of coherent text)
- Trinity falls back to the non-fused path, losing the performance benefit

## The Fix

We patch three libraries to thread `expert_bias` through the fused kernel pipeline:

1. **nki-library** (`router_topk.py`): Add `expert_bias` parameter. Load bias to SBUF, broadcast, add to affinities for top-K selection only. ~30 lines.
2. **neuronx-distributed** (`routing.py` + `moe_fused_tkg.py`): Add `expert_bias_size` to `RouterTopK`, pass bias through `optional_kwargs` to fused kernel. ~15 lines.
3. **neuronx-distributed-inference** (`moe_v2.py`): Pass `expert_bias_size` from model config to `RouterTopK`. ~3 lines.

All changes are on `feature/expert-bias-support` branches in our forks.

## This Notebook

This notebook:
1. **Patches** the three libraries from our fork branches
2. **Validates correctness** — first-token output with fused kernel matches non-fused
3. **Benchmarks** — measures TKG latency improvement from fused kernel
4. Tests on Trinity-Nano (quick validation) and Trinity-Mini (real benchmark)

### Prerequisites

- trn2.3xlarge instance with SDK 2.28 DLAMI (LNC=2, default)
- Neuron venv: `/opt/aws_neuronx_venv_pytorch_inference_vllm_0_13/`
- Model weights downloaded to `/mnt/models/`
- NxDI cloned to `/home/ubuntu/nxdi/` (`git clone https://github.com/aws-neuron/neuronx-distributed-inference.git /home/ubuntu/nxdi`)

### LNC Constraint

With LNC=2 (default) on trn2.3xlarge, there are 4 logical NeuronCores. The runtime requires `NEURON_RT_NUM_CORES` to be 1 or the full device (4). **TP=2 is NOT valid with LNC=2.** This notebook uses TP=1 for Nano and TP=4 for Mini.

### Running This Notebook

**Important:** NeuronCore device memory is not reliably released within a single kernel session via `del model; gc.collect()`. You **must restart the kernel** between model loads (e.g., between Nano and Mini sections, and between Mini non-fused and Mini fused). Each section that loads a model is marked with a restart reminder.

### Validated Results (2026-03-18, trn2.3xlarge, SDK 2.28)

- **Trinity-Nano (TP=1):** 5/5 first-token matches between fused and non-fused
- **Trinity-Mini (TP=4, BS=1):** TKG 11.5 ms (non-fused) vs 9.0 ms (fused) = **21.8% reduction, +28% throughput**. 5/5 first-token correctness match.

## 0. Environment Setup

In [ ]:
import os
# NEURON_RT_NUM_CORES is set automatically by NxDI based on tp_degree.
# For Mini (TP=4), all 4 cores are used. For Nano (TP=1), 1 core is used.

# Verify SDK versions before patching
import pkg_resources
for pkg in ['neuronx-cc', 'neuronx-distributed', 'neuronx-distributed-inference', 'torch-neuronx', 'torch']:
    try:
        print(f"{pkg}: {pkg_resources.get_distribution(pkg).version}")
    except pkg_resources.DistributionNotFound:
        print(f"{pkg}: not installed")

!neuron-ls | head -5

## 1. Install Patched Libraries

We install our forked versions of the three libraries from their `feature/expert-bias-support` branches.

**What each patch does:**

### nki-library (router_topk.py)
Adds `expert_bias` parameter to the `router_topk` NKI kernel. When provided:
- Loads `expert_bias [1, E]` from HBM to SBUF via `nisa.dma_copy`
- Broadcasts to `[tile_size, E]` using `stream_shuffle_broadcast`
- Creates biased copy: `topk_input = affinities + expert_bias` (for selection)
- Original unbiased affinities preserved for routing weight scatter

### neuronx-distributed (routing.py + moe_fused_tkg.py)
- `RouterTopK` gains `expert_bias_size` param and registers a buffer for the bias weights
- `MoEFusedTKG._moe_fused_tkg_kernel()` passes `expert_bias` through `optional_kwargs`

### neuronx-distributed-inference (moe_v2.py)
- `initialize_moe_module()` passes `expert_bias_size` from model config to `RouterTopK`

In [ ]:
%%bash
set -e

echo "=== Installing patched nki-library ==="
# pip install fails due to setuptools_scm issue -- clone and copy the patched file directly
if [ ! -d /tmp/nki-library ]; then
    git clone -b feature/expert-bias-support https://github.com/jimburtoft/nki-library.git /tmp/nki-library 2>&1 | tail -2
fi
NKILIB_SITE=$(python -c "import nkilib; print(nkilib.__path__[0])")
cp /tmp/nki-library/src/nkilib_src/nkilib/core/router_topk/router_topk.py $NKILIB_SITE/core/router_topk/router_topk.py
echo "  Copied patched router_topk.py to $NKILIB_SITE/core/router_topk/"

echo ""
echo "=== Installing patched neuronx-distributed ==="
pip install --no-deps git+https://github.com/jimburtoft/neuronx-distributed.git@feature/expert-bias-support 2>&1 | tail -3

echo ""
echo "=== Installing patched neuronx-distributed-inference ==="
pip install --no-deps git+https://github.com/jimburtoft/neuronx-distributed-inference.git@feature/expert-bias-support 2>&1 | tail -3

echo ""
echo "=== Done. Verifying installations ==="
pip show neuronx-distributed 2>/dev/null | grep -E 'Version|Location'
pip show neuronx-distributed-inference 2>/dev/null | grep -E 'Version|Location'

### Verify the patches are active

Check that `RouterTopK` now accepts `expert_bias_size` and that `moe_v2.initialize_moe_module` passes it through.

In [ ]:
import inspect

# Check RouterTopK has expert_bias_size parameter
from neuronx_distributed.modules.moe.routing import RouterTopK
sig = inspect.signature(RouterTopK.__init__)
assert 'expert_bias_size' in sig.parameters, "RouterTopK missing expert_bias_size param!"
print(f"RouterTopK.__init__ params: {list(sig.parameters.keys())}")
print("  -> expert_bias_size: PRESENT")

# Check MoEFusedTKG passes expert_bias
from neuronx_distributed.modules.moe.moe_fused_tkg import MoEFusedTKG
source = inspect.getsource(MoEFusedTKG._moe_fused_tkg_kernel)
assert 'expert_bias' in source, "MoEFusedTKG._moe_fused_tkg_kernel missing expert_bias!"
print(f"MoEFusedTKG._moe_fused_tkg_kernel: expert_bias plumbing PRESENT")

# Check NxDI moe_v2 passes expert_bias_size
from neuronx_distributed_inference.modules.moe_v2 import initialize_moe_module
source = inspect.getsource(initialize_moe_module)
assert 'expert_bias_size' in source, "moe_v2.initialize_moe_module missing expert_bias_size!"
print(f"moe_v2.initialize_moe_module: expert_bias_size plumbing PRESENT")

print("\nAll patches verified.")

## 2. Download Model Weights

Download Trinity-Nano and Trinity-Mini weights if not already present.

In [ ]:
%%bash
set -e

# Set your HuggingFace token before running: export HF_TOKEN=hf_...
if [ -z "$HF_TOKEN" ]; then
    echo "ERROR: Set HF_TOKEN environment variable first: export HF_TOKEN=hf_..."
    exit 1
fi

# Trinity-Nano
if [ ! -d "/mnt/models/Trinity-Nano-HF" ]; then
    echo "Downloading Trinity-Nano..."
    huggingface-cli download arcee-ai/Trinity-Nano-Preview \
        --local-dir /mnt/models/Trinity-Nano-HF \
        --token "$HF_TOKEN" 2>&1 | tail -3
else
    echo "Trinity-Nano already present at /mnt/models/Trinity-Nano-HF"
    ls /mnt/models/Trinity-Nano-HF/*.safetensors | wc -l | xargs -I{} echo "  {} safetensor files"
fi

# Trinity-Mini
if [ ! -d "/mnt/models/Trinity-Mini-HF" ]; then
    echo "Downloading Trinity-Mini..."
    huggingface-cli download arcee-ai/Trinity-Mini \
        --local-dir /mnt/models/Trinity-Mini-HF \
        --token "$HF_TOKEN" 2>&1 | tail -3
else
    echo "Trinity-Mini already present at /mnt/models/Trinity-Mini-HF"
    ls /mnt/models/Trinity-Mini-HF/*.safetensors | wc -l | xargs -I{} echo "  {} safetensor files"
fi

## 3. Trinity-Nano Validation (TP=1)

Quick validation on the smaller model. Trinity-Nano has:
- ~6B total parameters, ~1B active per token
- 54 MoE layers, 128 experts, top_k=8, intermediate=256
- Nano's small intermediate size means the fused kernel won't show a throughput improvement,
  but it validates that expert_bias is working correctly.
- **TP=1** because LNC=2 (default) only allows TP=1 or TP=4 on trn2.3xlarge.

We compile and run both **non-fused** (default) and **fused** paths, comparing first-token output.

In [ ]:
import sys
import time
import gc
import torch

sys.path.insert(0, "/home/ubuntu/nxdi/contrib/models/Trinity/src")

from transformers import AutoTokenizer
from neuronx_distributed_inference.models.config import MoENeuronConfig
from modeling_trinity import NeuronTrinityForCausalLM, TrinityInferenceConfig

In [ ]:
NANO_PATH = "/mnt/models/Trinity-Nano-HF"
NANO_TP = 1
SEQ_LEN = 2048
BS = 1

TEST_PROMPTS = [
    "Hello, how are you?",
    "What is the capital of France?",
    "Explain quantum computing in simple terms.",
    "Write a Python function to compute fibonacci numbers.",
    "The meaning of life is",
]

tokenizer = AutoTokenizer.from_pretrained(NANO_PATH, trust_remote_code=True)
print(f"Tokenizer loaded. Vocab size: {tokenizer.vocab_size}")

### 3a. Compile and run NON-FUSED (baseline)

In [ ]:
# Non-fused config (default -- no moe_fused_nki_kernel_enabled)
neuron_config = MoENeuronConfig(
    tp_degree=NANO_TP,
    batch_size=BS,
    seq_len=SEQ_LEN,
    torch_dtype=torch.bfloat16,
    on_device_generation=True,
)
config = TrinityInferenceConfig.from_pretrained(NANO_PATH, neuron_config=neuron_config)
model_nonfused = NeuronTrinityForCausalLM(NANO_PATH, config)

compiled_nonfused = "/mnt/models/compiled-nano-nonfused"
if not os.path.exists(os.path.join(compiled_nonfused, "model.pt")):
    print("Compiling non-fused Nano...")
    t0 = time.time()
    model_nonfused.compile(compiled_nonfused)
    print(f"Compile time: {time.time() - t0:.1f}s ({(time.time() - t0)/60:.1f} min)")
else:
    print(f"Reusing compiled model at {compiled_nonfused}")

print("Loading non-fused model...")
t0 = time.time()
model_nonfused.load(compiled_nonfused)
print(f"Load time: {time.time() - t0:.1f}s")

In [ ]:
# Generate first tokens with non-fused model
nonfused_results = {}
for prompt in TEST_PROMPTS:
    model_nonfused.reset()
    inputs = tokenizer(prompt, return_tensors="pt")
    input_ids = inputs.input_ids
    seq_ids = torch.arange(BS)
    position_ids = torch.arange(input_ids.shape[1]).unsqueeze(0)

    with torch.no_grad():
        outputs = model_nonfused(input_ids, position_ids=position_ids, seq_ids=seq_ids)

    logits = outputs.logits if hasattr(outputs, 'logits') else outputs[0]
    top1_id = torch.argmax(logits[:, -1, :], dim=-1).item()
    top1_token = tokenizer.decode([top1_id])
    nonfused_results[prompt] = {'token_id': top1_id, 'token': top1_token}
    print(f"  [{prompt[:40]:40s}] -> '{top1_token}' (id={top1_id})")

# Clean up
del model_nonfused
gc.collect()
print("\nNon-fused model unloaded.")

### 3b. Compile and run FUSED (with expert_bias patch)

In [ ]:
# Fused config -- enable the fused MoE TKG kernel
neuron_config = MoENeuronConfig(
    tp_degree=NANO_TP,
    batch_size=BS,
    seq_len=SEQ_LEN,
    torch_dtype=torch.bfloat16,
    on_device_generation=True,
    moe_fused_nki_kernel_enabled=True,
)
config = TrinityInferenceConfig.from_pretrained(NANO_PATH, neuron_config=neuron_config)
model_fused = NeuronTrinityForCausalLM(NANO_PATH, config)

compiled_fused = "/mnt/models/compiled-nano-fused-eb"
if not os.path.exists(os.path.join(compiled_fused, "model.pt")):
    print("Compiling fused Nano (with expert_bias)...")
    t0 = time.time()
    model_fused.compile(compiled_fused)
    print(f"Compile time: {time.time() - t0:.1f}s ({(time.time() - t0)/60:.1f} min)")
else:
    print(f"Reusing compiled model at {compiled_fused}")

print("Loading fused model...")
t0 = time.time()
model_fused.load(compiled_fused)
print(f"Load time: {time.time() - t0:.1f}s")

In [ ]:
# Generate first tokens with fused model and compare
fused_results = {}
print(f"{'Prompt':40s} | {'Non-fused':15s} | {'Fused':15s} | Match")
print("-" * 85)

all_match = True
for prompt in TEST_PROMPTS:
    model_fused.reset()
    inputs = tokenizer(prompt, return_tensors="pt")
    input_ids = inputs.input_ids
    seq_ids = torch.arange(BS)
    position_ids = torch.arange(input_ids.shape[1]).unsqueeze(0)

    with torch.no_grad():
        outputs = model_fused(input_ids, position_ids=position_ids, seq_ids=seq_ids)

    logits = outputs.logits if hasattr(outputs, 'logits') else outputs[0]
    top1_id = torch.argmax(logits[:, -1, :], dim=-1).item()
    top1_token = tokenizer.decode([top1_id])
    fused_results[prompt] = {'token_id': top1_id, 'token': top1_token}

    nf = nonfused_results[prompt]
    match = "YES" if nf['token_id'] == top1_id else "NO"
    if nf['token_id'] != top1_id:
        all_match = False
    print(f"  {prompt[:38]:38s} | {repr(nf['token']):15s} | {repr(top1_token):15s} | {match}")

print()
if all_match:
    print("RESULT: All first tokens MATCH -- expert_bias is working correctly in fused kernel.")
else:
    print("RESULT: MISMATCH detected -- expert_bias may not be correctly applied.")

In [ ]:
# Also test multi-token generation to check for coherence
NUM_TOKENS = 16

print(f"Generating {NUM_TOKENS} tokens with fused model:\n")
for prompt in TEST_PROMPTS[:3]:
    model_fused.reset()
    inputs = tokenizer(prompt, return_tensors="pt")
    input_ids = inputs.input_ids
    seq_ids = torch.arange(BS)
    position_ids = torch.arange(input_ids.shape[1]).unsqueeze(0)

    generated_ids = []
    with torch.no_grad():
        # CTE
        outputs = model_fused(input_ids, position_ids=position_ids, seq_ids=seq_ids)
        logits = outputs.logits if hasattr(outputs, 'logits') else outputs[0]
        next_id = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
        generated_ids.append(next_id.item())

        # TKG
        cur_pos = input_ids.shape[1]
        for _ in range(NUM_TOKENS - 1):
            pos_ids = torch.full((BS, 1), cur_pos, dtype=torch.long)
            outputs = model_fused(next_id, position_ids=pos_ids, seq_ids=seq_ids)
            logits = outputs.logits if hasattr(outputs, 'logits') else outputs[0]
            next_id = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
            generated_ids.append(next_id.item())
            cur_pos += 1

    generated_text = tokenizer.decode(generated_ids)
    print(f"  Prompt: {prompt}")
    print(f"  Output: {generated_text}")
    print()

del model_fused
gc.collect()
print("Fused model unloaded.")

---
## ⚠️ Restart Kernel Before Continuing

**You must restart the Jupyter kernel now** (Kernel > Restart) before running the Mini sections below.

NeuronCore device memory is not released by `del model; gc.collect()`. If you skip this step, the Mini model will fail to load with an out-of-memory error.

After restarting, re-run the imports cell (Cell 10: `import sys, time, gc, torch...`) and then continue from Section 4.

## 4. Trinity-Mini Benchmark (TP=4)

Trinity-Mini is where the fused kernel shows real benefit:
- ~26B total parameters, ~4.5B active per token
- 54 MoE layers, 128 experts, top_k=8, intermediate=**1024** (4x Nano)
- The larger intermediate size means selective loading actually saves significant HBM bandwidth

Previous results (Task 17, without expert_bias — garbled output):
- Non-fused: 11.5 ms/tok TKG, 86.4 tok/s
- Fused: 9.0 ms/tok TKG, 110.3 tok/s (**27.7% improvement**)

Now we test with expert_bias enabled — expecting same performance but **correct output**.

In [ ]:
MINI_PATH = "/mnt/models/Trinity-Mini-HF"
MINI_TP = 4
SEQ_LEN = 2048
BS = 1
NUM_GEN_TOKENS = 32
NUM_WARMUP = 3
NUM_MEASURE = 10

tokenizer_mini = AutoTokenizer.from_pretrained(MINI_PATH, trust_remote_code=True)
print(f"Mini tokenizer loaded. Vocab size: {tokenizer_mini.vocab_size}")

### 4a. Compile and benchmark NON-FUSED Mini

In [ ]:
neuron_config = MoENeuronConfig(
    tp_degree=MINI_TP,
    batch_size=BS,
    seq_len=SEQ_LEN,
    torch_dtype=torch.bfloat16,
    on_device_generation=True,
)
config = TrinityInferenceConfig.from_pretrained(MINI_PATH, neuron_config=neuron_config)
model_mini_nf = NeuronTrinityForCausalLM(MINI_PATH, config)

compiled_mini_nf = "/mnt/models/compiled-mini-nonfused"
if not os.path.exists(os.path.join(compiled_mini_nf, "model.pt")):
    print("Compiling non-fused Mini...")
    t0 = time.time()
    model_mini_nf.compile(compiled_mini_nf)
    print(f"Compile time: {time.time() - t0:.1f}s ({(time.time() - t0)/60:.1f} min)")
else:
    print(f"Reusing compiled model at {compiled_mini_nf}")

print("Loading non-fused Mini...")
t0 = time.time()
model_mini_nf.load(compiled_mini_nf)
print(f"Load time: {time.time() - t0:.1f}s")

In [ ]:
def benchmark_generation(model, tokenizer, prompt, bs, num_gen_tokens, num_warmup, num_measure):
    """Benchmark CTE + TKG pipeline. Returns dict with timings and generated text."""
    inputs = tokenizer(prompt, return_tensors="pt")
    input_ids = inputs.input_ids.expand(bs, -1)
    prompt_len = input_ids.shape[1]
    seq_ids = torch.arange(bs)

    # Warmup CTE
    for _ in range(2):
        model.reset()
        position_ids = torch.arange(prompt_len).unsqueeze(0).expand(bs, -1)
        with torch.no_grad():
            model(input_ids, position_ids=position_ids, seq_ids=seq_ids)

    # Measure CTE
    cte_times = []
    for _ in range(num_measure):
        model.reset()
        position_ids = torch.arange(prompt_len).unsqueeze(0).expand(bs, -1)
        t0 = time.perf_counter()
        with torch.no_grad():
            outputs = model(input_ids, position_ids=position_ids, seq_ids=seq_ids)
        cte_times.append((time.perf_counter() - t0) * 1000)

    logits = outputs.logits if hasattr(outputs, 'logits') else outputs[0]
    next_id = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
    generated_ids = [next_id[0].item()]

    # Warmup TKG
    cur_pos = prompt_len
    for i in range(num_warmup):
        pos_ids = torch.full((bs, 1), cur_pos, dtype=torch.long)
        with torch.no_grad():
            outputs = model(next_id, position_ids=pos_ids, seq_ids=seq_ids)
        logits = outputs.logits if hasattr(outputs, 'logits') else outputs[0]
        next_id = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
        generated_ids.append(next_id[0].item())
        cur_pos += 1

    # Measure TKG
    tkg_times = []
    for i in range(num_gen_tokens - num_warmup - 1):
        pos_ids = torch.full((bs, 1), cur_pos, dtype=torch.long)
        t0 = time.perf_counter()
        with torch.no_grad():
            outputs = model(next_id, position_ids=pos_ids, seq_ids=seq_ids)
        tkg_times.append((time.perf_counter() - t0) * 1000)
        logits = outputs.logits if hasattr(outputs, 'logits') else outputs[0]
        next_id = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
        generated_ids.append(next_id[0].item())
        cur_pos += 1

    cte_median = sorted(cte_times)[len(cte_times) // 2]
    tkg_median = sorted(tkg_times)[len(tkg_times) // 2]
    text = tokenizer.decode(generated_ids)

    return {
        'cte_median_ms': cte_median,
        'tkg_median_ms': tkg_median,
        'tkg_tok_per_s': bs * 1000.0 / tkg_median,
        'generated_text': text,
        'first_token': tokenizer.decode([generated_ids[0]]),
        'first_token_id': generated_ids[0],
    }

In [ ]:
print("Benchmarking non-fused Mini BS=1...\n")
mini_nf_result = benchmark_generation(
    model_mini_nf, tokenizer_mini,
    "Hello, how are you?",
    bs=BS, num_gen_tokens=NUM_GEN_TOKENS,
    num_warmup=NUM_WARMUP, num_measure=NUM_MEASURE,
)
print(f"  CTE:  {mini_nf_result['cte_median_ms']:.1f} ms")
print(f"  TKG:  {mini_nf_result['tkg_median_ms']:.1f} ms/tok")
print(f"  Throughput: {mini_nf_result['tkg_tok_per_s']:.1f} tok/s")
print(f"  First token: {repr(mini_nf_result['first_token'])}")
print(f"  Generated: {mini_nf_result['generated_text'][:100]}")

# Save first token for comparison
mini_nf_first_tokens = {}
for prompt in TEST_PROMPTS:
    model_mini_nf.reset()
    inputs = tokenizer_mini(prompt, return_tensors="pt")
    seq_ids = torch.arange(BS)
    position_ids = torch.arange(inputs.input_ids.shape[1]).unsqueeze(0)
    with torch.no_grad():
        outputs = model_mini_nf(inputs.input_ids, position_ids=position_ids, seq_ids=seq_ids)
    logits = outputs.logits if hasattr(outputs, 'logits') else outputs[0]
    top1_id = torch.argmax(logits[:, -1, :], dim=-1).item()
    mini_nf_first_tokens[prompt] = top1_id

del model_mini_nf
gc.collect()
print("\nNon-fused Mini unloaded.")

---
## ⚠️ Restart Kernel Before Continuing

**Restart the kernel again** (Kernel > Restart) before loading the fused Mini model.

After restarting, re-run: the imports cell, the Mini config cell (MINI_PATH, tokenizer), and the `benchmark_generation` function cell. Then continue below.

### 4b. Compile and benchmark FUSED Mini (with expert_bias)

In [ ]:
neuron_config = MoENeuronConfig(
    tp_degree=MINI_TP,
    batch_size=BS,
    seq_len=SEQ_LEN,
    torch_dtype=torch.bfloat16,
    on_device_generation=True,
    moe_fused_nki_kernel_enabled=True,
)
config = TrinityInferenceConfig.from_pretrained(MINI_PATH, neuron_config=neuron_config)
model_mini_fused = NeuronTrinityForCausalLM(MINI_PATH, config)

compiled_mini_fused = "/mnt/models/compiled-mini-fused-eb"
if not os.path.exists(os.path.join(compiled_mini_fused, "model.pt")):
    print("Compiling fused Mini (with expert_bias)...")
    t0 = time.time()
    model_mini_fused.compile(compiled_mini_fused)
    print(f"Compile time: {time.time() - t0:.1f}s ({(time.time() - t0)/60:.1f} min)")
else:
    print(f"Reusing compiled model at {compiled_mini_fused}")

print("Loading fused Mini...")
t0 = time.time()
model_mini_fused.load(compiled_mini_fused)
print(f"Load time: {time.time() - t0:.1f}s")

In [ ]:
print("Benchmarking fused Mini BS=1...\n")
mini_fused_result = benchmark_generation(
    model_mini_fused, tokenizer_mini,
    "Hello, how are you?",
    bs=BS, num_gen_tokens=NUM_GEN_TOKENS,
    num_warmup=NUM_WARMUP, num_measure=NUM_MEASURE,
)
print(f"  CTE:  {mini_fused_result['cte_median_ms']:.1f} ms")
print(f"  TKG:  {mini_fused_result['tkg_median_ms']:.1f} ms/tok")
print(f"  Throughput: {mini_fused_result['tkg_tok_per_s']:.1f} tok/s")
print(f"  First token: {repr(mini_fused_result['first_token'])}")
print(f"  Generated: {mini_fused_result['generated_text'][:100]}")

In [ ]:
# Correctness check -- compare first tokens
print(f"{'Prompt':40s} | {'Non-fused':12s} | {'Fused':12s} | Match")
print("-" * 80)

mini_all_match = True
for prompt in TEST_PROMPTS:
    model_mini_fused.reset()
    inputs = tokenizer_mini(prompt, return_tensors="pt")
    seq_ids = torch.arange(BS)
    position_ids = torch.arange(inputs.input_ids.shape[1]).unsqueeze(0)
    with torch.no_grad():
        outputs = model_mini_fused(inputs.input_ids, position_ids=position_ids, seq_ids=seq_ids)
    logits = outputs.logits if hasattr(outputs, 'logits') else outputs[0]
    fused_id = torch.argmax(logits[:, -1, :], dim=-1).item()
    nf_id = mini_nf_first_tokens[prompt]

    nf_tok = tokenizer_mini.decode([nf_id])
    fused_tok = tokenizer_mini.decode([fused_id])
    match = "YES" if nf_id == fused_id else "NO"
    if nf_id != fused_id:
        mini_all_match = False
    print(f"  {prompt[:38]:38s} | {repr(nf_tok):12s} | {repr(fused_tok):12s} | {match}")

print()
if mini_all_match:
    print("RESULT: All Mini first tokens MATCH -- expert_bias fused kernel is correct.")
else:
    print("RESULT: MISMATCH detected on Mini.")

### 4c. Mini Results Summary

In [ ]:
print("=" * 70)
print("TRINITY-MINI FUSED MoE TKG BENCHMARK (TP=4, BS=1, seq_len=2048)")
print("=" * 70)
print(f"")
print(f"{'Metric':25s} | {'Non-fused':15s} | {'Fused+expert_bias':18s} | {'Change':10s}")
print("-" * 75)

cte_change = (mini_fused_result['cte_median_ms'] / mini_nf_result['cte_median_ms'] - 1) * 100
tkg_change = (mini_fused_result['tkg_median_ms'] / mini_nf_result['tkg_median_ms'] - 1) * 100
tp_change = (mini_fused_result['tkg_tok_per_s'] / mini_nf_result['tkg_tok_per_s'] - 1) * 100

print(f"{'CTE (ms)':25s} | {mini_nf_result['cte_median_ms']:15.1f} | {mini_fused_result['cte_median_ms']:18.1f} | {cte_change:+.1f}%")
print(f"{'TKG (ms/tok)':25s} | {mini_nf_result['tkg_median_ms']:15.1f} | {mini_fused_result['tkg_median_ms']:18.1f} | {tkg_change:+.1f}%")
print(f"{'Throughput (tok/s)':25s} | {mini_nf_result['tkg_tok_per_s']:15.1f} | {mini_fused_result['tkg_tok_per_s']:18.1f} | {tp_change:+.1f}%")
print(f"{'First token match':25s} | {'--':15s} | {'YES' if mini_all_match else 'NO':18s} | {'--':10s}")
print()
print(f"Non-fused output: {mini_nf_result['generated_text'][:80]}")
print(f"Fused output:     {mini_fused_result['generated_text'][:80]}")

del model_mini_fused
gc.collect()
print("\nFused Mini unloaded.")

## 5. Summary

### What was patched

| Repo | File | Change | Lines |
|------|------|--------|-------|
| nki-library | `router_topk.py` | Add `expert_bias` param, load/broadcast/add in SBUF | ~30 |
| neuronx-distributed | `routing.py` | Add `expert_bias_size` to `RouterTopK`, register buffer | ~10 |
| neuronx-distributed | `moe_fused_tkg.py` | Pass `expert_bias` through `optional_kwargs` | ~6 |
| neuronx-distributed-inference | `moe_v2.py` | Pass `expert_bias_size` from config | ~3 |

### Branches

| Repo | Branch |
|------|--------|
| [nki-library](https://github.com/jimburtoft/nki-library/tree/feature/expert-bias-support) | `feature/expert-bias-support` |
| [neuronx-distributed](https://github.com/jimburtoft/neuronx-distributed/tree/feature/expert-bias-support) | `feature/expert-bias-support` |
| [neuronx-distributed-inference](https://github.com/jimburtoft/neuronx-distributed-inference/tree/feature/expert-bias-support) | `feature/expert-bias-support` |

### To use in production

```bash
# nki-library: pip install fails due to setuptools_scm -- clone and copy instead
git clone -b feature/expert-bias-support https://github.com/jimburtoft/nki-library.git /tmp/nki-library
NKILIB_SITE=$(python -c "import nkilib; print(nkilib.__path__[0])")
cp /tmp/nki-library/src/nkilib_src/nkilib/core/router_topk/router_topk.py $NKILIB_SITE/core/router_topk/router_topk.py

# neuronx-distributed and neuronx-distributed-inference
pip install --no-deps git+https://github.com/jimburtoft/neuronx-distributed.git@feature/expert-bias-support
pip install --no-deps git+https://github.com/jimburtoft/neuronx-distributed-inference.git@feature/expert-bias-support
```

Then set `moe_fused_nki_kernel_enabled=True` in `MoENeuronConfig`.